# Tech Talent Recruiting with Regex

![](image.jpg)

## 📖 The Background
As a data analyst at a top global HR consultancy, you have the important task: to search through a vast resumes dataset and find candidates for tech jobs.  
This project is not just a chance for you to apply your skills in regular expressions and data preprocessing, but it also plays a pivotal role in matching suitable candidates with the right jobs.

As a Data Analyst at a leading global HR consultancy, your mission is to delve into an extensive database of resumes to identify suitable candidates for tech-focused roles. This task involves using regular expressions to extract key data points and applying data preprocessing techniques to organize this information effectively.

## 💾 The Data

| Column      | Data Type | Description                                                  |
|-------------|-----------|--------------------------------------------------------------|
| `ID`        | float     | Unique identifier for each resume.                           |
| `Resume_str`| object    | Full text of the resume, rich with details for analysis.     |
| `Category`  | object    | Job category of the resume, indicating the field of expertise. |

# Objectives
Apply your regular expressions and data preprocessing skills to efficiently organize the `resumes` data.
1) Generate a new DataFrame named `candidates_df` containing the following columns:
- `id`: Candidate ID.
- `job_title`: Most recent job title.
- `tech_skills`: List of technical skills that may include Python, SQL, R, and Excel.
- `education`: Highest educational degree such as PhD, Master, or Bachelor.

⚠️ Keep only records that are complete and do not contain null or empty string values across these columns!

### Importing

In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
#Import the data
orig_df = pd.read_csv('resumes.csv')
#Inspect the data
display(orig_df.head(5))
display(orig_df.info())

,ID,Resume_str,Category
0,16852973.0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,22323967.0,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,33176873.0,HR DIRECTOR Summary Over 2...,HR
3,27018550.0,HR SPECIALIST Summary Dedica...,HR
4,17812897.0,HR MANAGER Skill Highlights ...,HR


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1352 entries, 0 to 1351
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          1352 non-null   float64
 1   Resume_str  1352 non-null   object 
 2   Category    1352 non-null   object 
dtypes: float64(1), object(2)
memory usage: 31.8+ KB


None

In [3]:
#Inspect values across variables
display(orig_df.iloc[:,2].value_counts(dropna=False))

Category
INFORMATION-TECHNOLOGY    119
BUSINESS-DEVELOPMENT      119
FITNESS                   117
ADVOCATE                  115
SALES                     115
CONSULTANT                115
HEALTHCARE                113
HR                        108
DESIGNER                  106
TEACHER                   102
DIGITAL-MEDIA              95
AGRICULTURE                63
AUTOMOBILE                 35
BPO                        21
CHEF                        9
Name: count, dtype: int64

The dataset has 1,352 entries & no missing values. Each variable was explored briefly to get a better understanding of their values.
- **ID**:
    - No missing values. ID's are between 10,005,171.0 - 99,714,410.0.
- **Resume_str:**
    - No missing values. This variable is where most of the analysis (preprocessing) will be.
- **Category:**
    - No missing values. There are 15 unique categories each of which accounts for between nine to 119 resumes. There don't appear to be any typos, entry errors, or inconsistencies across these values, so no cleaning will be necessary for this variable.

Other than the preprocessing, no cleaning is necessary.

# Analysis I
Want to efficiently organize the dataset.
1) Generate a new DataFrame named `candidates_df` containing the following columns:
- `id`: Candidate ID.
- `job_title`: Most recent job title.
- `tech_skills`: List of technical skills that may include Python, SQL, R, and Excel.
- `education`: Highest educational degree such as PhD, Master, or Bachelor.

⚠️ Keep only records that are complete and do not contain null or empty string values across these columns.

The candidate ID can be retrieved using the 'ID' variable from the original dataset.  
When it comes to the 'Resume_str' variable, it needs to be established if the values are generally formatted the same.

In [212]:
#Create the dataframe & the 'id' variable
candidates_df_orig = pd.DataFrame(data=orig_df['ID'])
#Inspect the 'Resume_str' variable. Explore the resumes of 5 random people.
    #indexes explored: 0, 199, 527, 999, 1240
#display(orig_df.loc[0,'Resume_str'])

Upon investigating the 'Resume_str' variable, there is not a clear-cut format in how the relevant variables & details are listed. In five random resumes, this variable contained various subsections that were referred to as: "Summary," "Executive Profile," "Highlights," "Accomplishments," "Experience," "Education," "Skills," "Languages," "Interests," "Additional Information," "Skill Highlights," "Professional Experience," "Core Accomplishments," & more. Across these headings, there are some inconsistencies across various resumes. This makes processing this variable more challenging, so several assumptions will have to be made when applying regular expressions. Nevertheless, there are still some keywords, such as "Education" & "Experience," that will be useful in obtaining the relevant data.
- Although it is not listed anywhere, the most recent job title is the fully capitalized position listed at the beginning of every resume.

Furthermore, regular expressions need to be built that can find matches regardless of capitalization & the neighboring characters. The "\b" anchor can be used to match characters at the start or end of a word. This will help to find matches that are surrounded by white spaces, tabs, or punctuation characters (e.g. commas, periods).  
To handle capitalization, all possible matches could be included in the regex pattern. Alternatively, the `flags` parameter in some regex functions can be specified to ignore case sensitivity when matching strings; set flags=re.IGNORECASE.

##### Most Recent Job Title
- To find the applicant's most recent job title, a regex pattern needs to be built to encapsulate the first set of words (or word) at the beginning of each resume, which is coincidentally fully capitalized. In some cases, this job position contains punctuation & other characters like parentheses. To search for fully capitalized words at the beginning of a string, the "^[A-Z]+" pattern can be used. The plus character is important because it searches for one or more such matches in case multiple words are in the specified position. To match for other cases, various metacharacters could be used like "\s" (whitespaces), "\." (periods), "\," (commas), etc.
    - As an example of what this pattern may encounter, the beginning of the resume at the index 1252 is: "GLOBAL DIGITAL SERVICING ~ DIGITAL ONBOARDING COORDINATOR (CONTRACTOR)".
- Since the `re.search()` function scans a string for the first occurrence of a specified pattern, it would be more appropriate to find this information.
##### Technical Skills
- To find listed technical skills, a regex pattern can be built that searches for keywords relating to such skills. Since multiple terms are being searched for, the `re.findall()` function would be more appropriate because it can return all matches. To start, the following skills will be used: R, SQL, Python, Excel. Others may be added.
    - Note that this function returns all matches in a list. Even if no matches are found, an empty list is returned.
##### Highest Educational Degree
- To find the highest educational degree achieved by an applicant, a similar process can be used as for the technical skills. By supplying various matches, using `re.findall()`, regex can retrieve them all. From there, it can be determined which education is the highest by referencing education levels. To start, the following degrees & education levels will be used: PhD, Doctoral, Masters, Bachelors, Associates, High School. Others may be added.
    - For reference, degrees from highest to lowest follow as: Doctoral/PhD, Masters, Bachelors, Associates, High School, etc.
    - In technical areas, other degrees could include the following: MCS, BCS (Master & Bachelor of Computer Science respectively).

Moreover, an understanding of the relevant regex functions are important. If no matches are found, `re.search()` will output a "NoneType" object. Conversely, `re.findall()` will output an empty list. When no matches to a regular expression are found, the result needs to be an empty string (""). As such, the code will need to specify this as the result when no matches are found.  
Additionally, if multiple matches are found, then they should be joined together with a comma & a space; e.g. "skill1, skill2".
- To retrieve the matching substrings from `re.search()`, need to append the `.group()` method.

In [213]:
#Build a regex pattern to look for the most recent job title
pattern_recent_job = r"^\s*([A-Z\.\,\-\~\/\(\)\s]+?)(?=\n|\\n|\s{2,}|$)"
    #the ^ specifies to search at the beginning of the string
    #the \s refers to a whitespace; the * indicates that the preceding character(s) will occur 0 or more times
    #the () are used to indicate the capture group; the substring to be extracted
    #the [] encapsulates character sets, anchors, & characters
    #[A-Z] looks for capital letters
    #\. searches for a period; this is similar for other characters including: ",", "-", "~", "/", "(", ")"
    #+? matches the specified pattern a minimum number of times that it occurs; a "lazy" quantifier
    #?= looks forward to later patterns without capturing them
    #the sequence "\n|\nn" looks for newlines &/or literal "\n" characters
        #the | searches for multiple options; \n or \\n
    #\s{2,} looks for 2 or more whitespaces
    #$ specifies to search at the end of the string

#Build a regex pattern to match technical skills
pattern_tech_skill = r"\b(r|sql|python|excel)\b"
    #the \b searches for matches at the start/end of a word; is an convenient way to search for substrings without directly 
        # specifying surrounding characters that are not wanted

#Build a regex pattern to look for all educational degrees
pattern_educ = r"\b(phd|doctoral|master|masters|MCS|BCS|bachelor|bachelors|associates|high school)\b"
    #need to consider if such strings could come up in a resume in a way that doesn't refer to a person's education.
        #e.g. "associate" could refer to an associate-like position. For this reason, "associate" was excluded.

In [214]:
#Create lists to store the extracted information
recent_jobs = []
tech_skills = []
educ_levels = []
uniq_educ_orig = []

#Loop through each applicant's resume
for res in orig_df['Resume_str']:
    #RECENT JOB POSITION
    #use the regex pattern to look for a match
    job_match = re.search(pattern_recent_job, res)
    #if a match is found
    if job_match is not None:
        job_match = job_match.group(1).strip()
    #if no match is found, use an empty string
    else:
        job_match = ""
    #append the result to the recent_jobs list
    recent_jobs.append(job_match)
    
    
    #TECH SKILLS
    #use the regex pattern to look for matches
    tech_skills_matches = re.findall(pattern_tech_skill, res, flags=re.IGNORECASE)
    #format the capitalization of each match. add a unique case for "SQL" which should be fully capitalized
    tech_skills_matches = [i.title() if i != 'SQL' else i.upper() for i in tech_skills_matches]
    #select the unique tech skills
    tech_skills_matches = list(set(tech_skills_matches))
    #append the matches to the tech_skills list, each formatted as a comma-separated string
    tech_skills.append(", ".join(tech_skills_matches))
    
    
    #EDUCATION LEVEL(S)
    #use the regex pattern to look for matches
    educ_lev_matches = re.findall(pattern_educ, res, flags=re.IGNORECASE)    
    #format the capitalization of each match
    educ_lev_matches = [i.title() for i in educ_lev_matches]
    #select the unique education levels
    educ_lev_matches = list(set(educ_lev_matches))
    #obtain the unique education levels
    uniq_educ_orig.append(educ_lev_matches)
    #append the matches to the educ_levels list, each formatted as a comma-separated string
    educ_levels.append(", ".join(educ_lev_matches))

#display(recent_jobs)

In [216]:
#Put the resulting lists into a data frame
candidates_df = candidates_df_orig.copy()
candidates_df['job_title'] = recent_jobs
candidates_df['tech_skills'] = tech_skills
candidates_df['educ_levels'] = educ_levels

#Only want records with complete values across all variables. Remove those with missing values
    #This reduces the dataset from 1,352 data points to 444
candidates_df = candidates_df[
    (candidates_df['job_title'] != '') & (candidates_df['tech_skills'] != '') & (candidates_df['educ_levels'] != '')]
#display(candidates_df)

In [217]:
#Obtain all the unique matches found in the 'educ_levels' variable
uniq_educ = sorted(list(set([edu for element in uniq_educ_orig for edu in element])))
#display(uniq_educ)

The next step in this process involves determining the highest educational degree achieved in each resume. So far, the 'educ_levels' variable in the `candidates_df` data frame contains all degrees obtained (or the highest level of education accomplished). Many terms were captured across multiple resumes, so each comma-separated list needs to be evaluated for the highest education level. This could be performed in a multitude of ways including complex sorting methods or for loops.

In the cell directly above, it was determined that the unique educational levels include: "Associates," "Bachelor/Bachelors," "Bcs," "Doctoral," "High School," "Master/Masters," "Mcs," & "Phd." From a hierarchical standpoint, the order of these degrees, from highest to lowest, would be: "Doctoral/PhD," "Master/Masters/MCS," "Bachelor/Bachelors/BCS," "Associates," "High School." Given the nature of these strings, there is no easy way to sort them according to their prowess.  
In this case, a nested for loop was utilized. It needs to consider cases where either multiple education-level matches or only a single match was found.

In [218]:
#Want to truncate the 'educ_levels' variable to obtain the highest educational degree achieved
#loop through the dataset by row
for i in candidates_df.index:
    #cases where there are multiple education-level matches
    if ", " in candidates_df.loc[i,'educ_levels']:
        #search for Doctoral/PhD matches (highest)
        if 'Doctoral' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'Doctoral'
        elif 'Phd' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'PhD'
        
        #search for Master/Masters/Mcs (2nd highest)
        elif 'Master' in candidates_df.loc[i,'educ_levels'] or 'Masters' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'Masters'
        elif 'Mcs' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'MCS'
        
        #search for Bachelor/Bachelors/BCS (3rd highest)
        elif 'Bachelor' in candidates_df.loc[i,'educ_levels'] or 'Bachelors' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'Bachelors'
        elif 'Bcs' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'BCS'
        
        #search for Associates (4th highest)
        elif 'Associates' in candidates_df.loc[i,'educ_levels']:
            candidates_df.loc[i,'education'] = 'Associates'
        #search for High School (5th highest)
        else:
            candidates_df.loc[i,'education'] = 'High School'
    
    
    #cases where there is only 1 education-level match -- no changes necessary
    else:
        candidates_df.loc[i,'education'] = candidates_df.loc[i,'educ_levels']

#Check the code (444 rows)
display(candidates_df.sample(10, random_state=10))

,ID,job_title,tech_skills,educ_levels,education
1127,29723311.0,IT CONSULTANT,Excel,"Master, High School, Bachelor",Masters
1078,55097118.0,SALES ASSOCIATE,Excel,"High School, Bachelor",Bachelors
698,98300955.0,"PLATFORM ARCHITECT, HEALTHCARE INCUBATION LAB ...",SQL,Bachelor,Bachelor
624,24412546.0,BUSINESS DEVELOPMENT ASSOCIATE,"R, Excel",Master,Master
596,26278597.0,BUSINESS DEVELOPMENT MANAGER,"R, Excel",Bachelor,Bachelor
916,62994611.0,SOFTWARE DEVELOPER,"Python, SQL","Master, Bachelor",Masters
827,17431847.0,REHABILITATION SPECIALIST / MASSAGE THERAPIST,Excel,"Bachelor, Masters",Masters
450,49475708.0,CUSTOMER ADVOCATE,R,High School,High School
456,14064815.0,FAMILY ADVOCATE,Excel,"Master, Bachelor",Masters
95,27496514.0,HR CUSTOMER SERVICE REPRESENTATIVE,Excel,Master,Master


For the final submission, the 'educ_levels' variable should be removed beforehand.